# TDC-KV Actual Controlled Testing

This notebook runs the current implementation on GSM8K, controlled NIAH, and HotpotQA in guarded stages. Run cells in order and run each dataset pilot separately.

These are controlled implementation results, not yet direct ChunkKV table reproductions: GSM8K is currently zero-shot rather than the ChunkKV few-shot prompt, NIAH uses deterministic key retrieval rather than an LLM judge, and HotpotQA uses the original distractor split rather than LongBench HotpotQA.

## 1. Locate the repository and verify `branch-h`

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_NAME = "Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring"
REPO_URL = "https://github.com/JayGor-13/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring.git"
EXPECTED_BRANCH = "branch-h"

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content") / REPO_NAME,
]
REPO = next((path.resolve() for path in candidates if (path / "scripts/run_hf_grid.py").is_file()), None)

if REPO is None:
    target = Path("/content") / REPO_NAME
    subprocess.run(
        ["git", "clone", "--branch", EXPECTED_BRANCH, "--single-branch", REPO_URL, str(target)],
        check=True,
    )
    REPO = target.resolve()

os.chdir(REPO)
branch = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
revision = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("Repository:", REPO)
print("Branch:", branch)
print("Revision:", revision)
if branch != EXPECTED_BRANCH:
    raise RuntimeError(f"Expected {EXPECTED_BRANCH!r}, found {branch!r}.")

## 2. Install project dependencies without replacing Colab PyTorch

In [ ]:
packages = [
    "transformers>=4.43,<6",
    "datasets>=5.0.1",
    "accelerate>=1.14.0",
    "numpy>=2.0",
    "scipy>=1.13",
    "matplotlib>=3.8",
    "pandas>=2.2",
    "pytest>=8.4",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
print("Environment installation complete. Restart the kernel only if pip requested it.")

In [ ]:
import torch
import transformers

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required. Attach a T4 or better Colab GPU and restart.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA runtime:", torch.version.cuda)
subprocess.run(["nvidia-smi"], check=True, text=True)

## 3. Run preflight tests

Do not continue when this cell fails.

In [ ]:
preflight = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_hf_cache_e2e.py",
        "tests/test_hf_runner.py",
        "tests/test_eval_metrics.py",
        "tests/test_result_aggregation.py",
    ],
    cwd=REPO,
    text=True,
    timeout=900,
)
print("Return code:", preflight.returncode)
if preflight.returncode != 0:
    raise RuntimeError("Preflight tests failed. Do not run experiments.")

## 4. Define guarded experiment and validation helpers

In [ ]:
import json
import time
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = REPO / "outputs" / "actual_testing" / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
print("Run directory:", OUTPUT_DIR)

def validate_payload(payload, *, require_predictions=True):
    runs = payload.get("runs", [])
    if not runs:
        raise AssertionError("Result payload contains no runs.")
    errors = [run for run in runs if run.get("status") != "ok"]
    if errors:
        first = errors[0]
        raise AssertionError(
            f"{len(errors)} run(s) failed. First: {first.get('error_type')}: {first.get('error')}"
        )
    for run in runs:
        method = run.get("method")
        if require_predictions and not str(run.get("evicted_prediction", "")).strip():
            raise AssertionError(f"Empty prediction for {method} / {run.get('sample_id')}")
        if method != "fullkv":
            budget = int(run["config"]["budget"])
            if int(run["kept_tokens"]) > budget:
                raise AssertionError(f"Budget overflow in {run.get('sample_id')}")
            decode = run.get("decode_cache_summary") or {}
            if int(decode.get("budget_violations", 0)) != 0:
                raise AssertionError(f"Decode budget violation in {run.get('sample_id')}")
    return payload

def grouped_frame(payload):
    rows = []
    for group in payload.get("grouped_results", []):
        budget = group.get("budget", {})
        qa = group.get("qa_summary", {})
        cache = group.get("cache_summary", {})
        run_summary = group.get("run_summary", {})
        rows.append({
            "dataset": group.get("dataset"),
            "method": group.get("method"),
            "budget_type": budget.get("type"),
            "budget_value": budget.get("value"),
            "successful": run_summary.get("successful"),
            "failed": run_summary.get("failed"),
            "primary_metric": qa.get("primary_metric"),
            "primary_score": qa.get("primary_score"),
            "retention": cache.get("avg_retention_ratio"),
            "compression": cache.get("avg_compression_ratio"),
            "compression_multiplier": cache.get("avg_compression_multiplier"),
            "budget_gap": cache.get("avg_budget_gap"),
            "eviction_latency_ms": cache.get("avg_latency_ms"),
        })
    return pd.DataFrame(rows)

def run_grid(*, label, dataset_spec, budget_ratios, max_samples, max_length, max_new_tokens, recent_window, timeout_minutes):
    output = OUTPUT_DIR / f"{label}.json"
    command = [
        sys.executable, "-u", "scripts/run_hf_grid.py",
        "--models", MODEL,
        "--datasets", dataset_spec,
        "--methods", "fullkv,tdc_kv",
        "--budget-ratios", budget_ratios,
        "--thetas", "0.3",
        "--recent-windows", str(recent_window),
        "--alphas", "0.6",
        "--dependency-top-k", "8",
        "--max-chunk-tokens", "64",
        "--min-budget-utilization", "0.99",
        "--max-budget-shortfall-tokens", "1",
        "--prefill-block-size", "128",
        "--tier1-score-mode", "dependency",
        "--max-samples", str(max_samples),
        "--max-length", str(max_length),
        "--max-new-tokens", str(max_new_tokens),
        "--device", "cuda",
        "--dtype", "float16",
        "--allow-level2-fallback",
        "--progress",
        "--output", str(output),
    ]
    print("Command:", " ".join(command))
    started = time.perf_counter()
    try:
        completed = subprocess.run(
            command, cwd=REPO, text=True, timeout=timeout_minutes * 60
        )
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(f"{label} exceeded {timeout_minutes} minutes and was terminated.") from exc
    elapsed = time.perf_counter() - started
    print(f"{label}: return_code={completed.returncode}, elapsed={elapsed / 60:.1f} minutes")
    if completed.returncode != 0:
        raise RuntimeError(f"{label} failed with return code {completed.returncode}.")
    payload = validate_payload(json.loads(output.read_text(encoding="utf-8")))
    display(grouped_frame(payload))
    return payload


## 5. Guarded one-sample smoke tests

All three cells must pass before starting any pilot.

In [ ]:
gsm_smoke = run_grid(
    label="gsm8k_smoke",
    dataset_spec="name=gsm8k,source=openai/gsm8k,adapter=gsm8k,config=main,split=test,prompt_field=question,answer_field=answer",
    budget_ratios="0.75",
    max_samples=1,
    max_length=1024,
    max_new_tokens=32,
    recent_window=16,
    timeout_minutes=20,
)

In [ ]:
niah_smoke = run_grid(
    label="niah_smoke",
    dataset_spec="name=niah_512,source=niah,adapter=niah,context_length=512,needle_depth=0.5,seed=13",
    budget_ratios="0.75",
    max_samples=1,
    max_length=1536,
    max_new_tokens=16,
    recent_window=32,
    timeout_minutes=20,
)

In [ ]:
hotpot_smoke = run_grid(
    label="hotpotqa_smoke",
    dataset_spec="name=hotpotqa,source=hotpotqa/hotpot_qa,adapter=hotpotqa,config=distractor,split=validation,prompt_field=question,answer_field=answer,id_field=id",
    budget_ratios="0.75",
    max_samples=1,
    max_length=1024,
    max_new_tokens=16,
    recent_window=32,
    timeout_minutes=20,
)

## 6. Controlled pilots

Run these cells one at a time. The live CLI output identifies the active sample and budget. These pilots establish correctness and an initial quality trend; they are not final paper-scale experiments.

In [ ]:
gsm_pilot = run_grid(
    label="gsm8k_pilot",
    dataset_spec="name=gsm8k,source=openai/gsm8k,adapter=gsm8k,config=main,split=test,prompt_field=question,answer_field=answer",
    budget_ratios="0.75,0.5,0.25",
    max_samples=10,
    max_length=1024,
    max_new_tokens=128,
    recent_window=16,
    timeout_minutes=90,
)

In [ ]:
niah_pilot = run_grid(
    label="niah_pilot",
    dataset_spec="name=niah_1024_d50,source=niah,adapter=niah,context_length=1024,needle_depth=0.5,seed=13",
    budget_ratios="0.75,0.5,0.25",
    max_samples=5,
    max_length=2048,
    max_new_tokens=16,
    recent_window=32,
    timeout_minutes=60,
)

In [ ]:
hotpot_pilot = run_grid(
    label="hotpotqa_pilot",
    dataset_spec="name=hotpotqa,source=hotpotqa/hotpot_qa,adapter=hotpotqa,config=distractor,split=validation,prompt_field=question,answer_field=answer,id_field=id",
    budget_ratios="0.75,0.5,0.25",
    max_samples=5,
    max_length=2048,
    max_new_tokens=32,
    recent_window=32,
    timeout_minutes=60,
)

## 7. Combine successful pilot results and plot the quality/compression trend

In [ ]:
from benchmarks.eval_metrics import aggregate_grouped_runs

pilot_payloads = [
    value for name in ("gsm_pilot", "niah_pilot", "hotpot_pilot")
    if (value := globals().get(name)) is not None
]
if not pilot_payloads:
    raise RuntimeError("Run at least one pilot cell first.")

combined_runs = [run for payload in pilot_payloads for run in payload["runs"]]
combined = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "protocol": "controlled_pilot_not_paper_parity",
    "runs": combined_runs,
    "grouped_results": aggregate_grouped_runs(combined_runs),
}
combined_path = OUTPUT_DIR / "combined_pilots.json"
combined_path.write_text(json.dumps(combined, indent=2), encoding="utf-8")
combined_frame = grouped_frame(combined)
display(combined_frame.sort_values(["dataset", "method", "budget_value"], na_position="first"))
print("Combined results:", combined_path)

In [ ]:
import matplotlib.pyplot as plt

plot_rows = combined_frame.dropna(subset=["primary_score"]).copy()
plot_rows["retention_percent"] = plot_rows["retention"] * 100.0
datasets = list(plot_rows["dataset"].drop_duplicates())
fig, axes = plt.subplots(1, len(datasets), figsize=(5 * len(datasets), 4), squeeze=False)
for ax, dataset in zip(axes[0], datasets):
    subset = plot_rows[plot_rows["dataset"] == dataset]
    for method, method_rows in subset.groupby("method"):
        method_rows = method_rows.sort_values("retention_percent")
        ax.plot(method_rows["retention_percent"], method_rows["primary_score"], marker="o", label=method)
    ax.set_title(dataset)
    ax.set_xlabel("KV retention (%)")
    ax.set_ylabel("Dataset primary score")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()
plot_path = OUTPUT_DIR / "controlled_quality_vs_retention.png"
fig.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()
print("Plot saved to:", plot_path)

## 8. Archive this run

Keep the JSON files as the source of truth. Notebook tables and plots can always be regenerated from them.

In [ ]:
import shutil

archive_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Run archive:", archive_path)